# 04. Server key update (Sec. IV-F, Lemma 3)

Once per epoch the servers renew their key shares $k_i$ without the users. Every $S_j$ deals a fresh
random polynomial $[F]_j(X) = \sum_{k=1}^{t-1} \alpha_k X^k$ with constant term $0$ and sends
$F_{ji} = [F]_j(i)$ to each $S_i$. $S_i$ adds $\sum_j F_{ji}$ to $k_i$. Because every $[F]_j(0) = 0$, the
master key $K = \sum_i \lambda_i k_i$ does not move (Lemma 3), so $pk_u$, $K_u$ and every outsourced
document stay valid. This notebook shows the keys, the messages, the commitment check and the
user-side effect before and after one update, then a second update through the protocol runner.

In [1]:
import dataclasses
from collections.abc import Iterable, Sequence

from ris_sem_project import protocol
from ris_sem_project.params import PARAM_I, setup
from ris_sem_project.parties import ProtocolAbort
from ris_sem_project.parties.qpase_server import ShareOutcome
from ris_sem_project.parties.qpase_user import QPASEUser
from ris_sem_project.primitives.ring import Ring
from ris_sem_project.primitives.secret_sharing import reconstruct

PASSWORD = "correct horse battery staple"
DOCUMENTS = (  # (item id, keyword, server that stores it, document)
    (1, "medical", 1, b"Blood panel, 14 Sep 2026. Haemoglobin 14.1 g/dL, LDL 2.9 mmol/L, HbA1c 5.2%. No follow-up."),
    (2, "travel", 3, b"Booking KX42QP: Mumbai BOM to Lisbon LIS, 3 Nov 2026 09:40, seat 23A, one checked bag."),
    (3, "medical", 3, b"Lumbar spine MRI: mild disc bulge at L4-L5, no nerve compression. Physiotherapy advised."),
)


def peek(data: bytes, width: int = 8) -> str:
    text = data.hex()
    half = width // 2
    shown = text if len(text) <= width else f"{text[:half]}…{text[-half:]}"
    return f"{shown} ({len(data)} B)"


def payload_size(value: object) -> int:
    match value:
        case bytes():
            return len(value)
        case str():
            return len(value.encode())
        case tuple() | list():
            return sum(payload_size(v) for v in value)
        case _ if dataclasses.is_dataclass(value):
            return sum(payload_size(getattr(value, f.name)) for f in dataclasses.fields(value))
        case _:
            return 0


def table(headers: Sequence[str], rows: Iterable[Sequence[object]]) -> str:
    cells = [list(map(str, headers))] + [list(map(str, row)) for row in rows]
    widths = [max(len(row[c]) for row in cells) for c in range(len(headers))]
    lines = ["  ".join(cell.ljust(w) for cell, w in zip(row, widths)).rstrip() for row in cells]
    lines.insert(1, "  ".join("-" * w for w in widths))
    return "\n".join(lines)

## Setup (Sec. IV-A) and the server links

$N = 3$ servers, threshold $t = 2$. Besides the DKG key share, each server has an ML-DSA-44 key pair
$(pk_j^{sig}, sk_j)$ that signs its update shares, and `setup_servers` gives every server every
verification key. The paper assumes secure server-to-server channels (Sec. III-A). We stand in for
them with $E(i, \cdot)$ = AES-256-GCM under one random 32-byte key per server pair. That key is not part
of the scheme.

In [2]:
pp = setup(PARAM_I, N=3, t=2, mu=14)
ring = Ring(pp.q, pp.n)
servers = protocol.setup_servers(pp)
print(f"N = {pp.N}, t = {pp.t}, mu = {pp.mu}, n = {pp.n}, q = {pp.q}")
print(table(("server", "epoch", "ML-DSA-44 verification key", "linked servers (channel keys)"), [
    (f"S_{i}", server.epoch, peek(server.verification_key), sorted(server.peers))
    for i, server in servers.items()
]))

N = 3, t = 2, mu = 14, n = 512, q = 268435399
server  epoch  ML-DSA-44 verification key  linked servers (channel keys)
------  -----  --------------------------  -----------------------------
S_1     0      74d1…cb0e (1312 B)          [1, 2, 3]
S_2     0      e371…a079 (1312 B)          [1, 2, 3]
S_3     0      cf64…f572 (1312 B)          [1, 2, 3]


## Before the update: a registered user with documents

alice registers, logs in on $\{S_1, S_2\}$ and on $\{S_2, S_3\}$ (each login adds 1 to $\mu_u$ on the
contacted servers), and outsources three documents to $S_1$ and $S_3$.

In [3]:
alice = QPASEUser(pp, "alice", PASSWORD)
registered = protocol.register(alice, servers)
login_12_before = protocol.login(alice, servers, (1, 2))
login_23_before = protocol.login(alice, servers, (2, 3))
for item_id, keyword, server_id, document in DOCUMENTS:
    protocol.outsource(alice, servers[server_id], keyword, document, item_id)
print(f"K_u at Register:         {peek(registered.k_u)}")
print(f"K_u at Login {{1, 2}}:     {peek(login_12_before.k_u)}")
print(f"K_u at Login {{2, 3}}:     {peek(login_23_before.k_u)}")
print(f"pk_u:                    {peek(registered.key_message.user_pk)}")
print(table(("item", "keyword", "stored at", "|Ct|", "|C|"), [
    (i, keyword, f"S_{s}", len(item.index_ct), len(item.data_ct))
    for i, keyword, s, _ in DOCUMENTS
    for item in servers[s].storage["alice"] if item.item_id == i
]))

K_u at Register:         f2b1…ae43 (1792 B)
K_u at Login {1, 2}:     f2b1…ae43 (1792 B)
K_u at Login {2, 3}:     f2b1…ae43 (1792 B)
pk_u:                    4d2f…b686 (1312 B)
item  keyword  stored at  |Ct|  |C|
----  -------  ---------  ----  ---
1     medical  S_1        4256  118
2     travel   S_3        4256  114
3     medical  S_3        4256  116


The state of every server before the update. $K$ is shown only to demonstrate Lemma 3: it is
$\sum_{i \in T} \lambda_i k_i$ for the integer-coefficient subsets $T = \{1, 2\}$ and $T = \{2, 3\}$, and no server
or user ever computes it in the protocol.

In [4]:
def coefficients(poly):
    centered = ring.centered(poly)
    return f"{centered[:6].tolist()}..., max |coefficient| = {abs(centered).max()}"


def snapshot():
    return {
        "epoch": {i: s.epoch for i, s in servers.items()},
        "k": {i: s.key.k_i.copy() for i, s in servers.items()},
        "pk": {i: s.key.pk_i.copy() for i, s in servers.items()},
        "mu": {i: s.users["alice"].fail_count for i, s in servers.items()},
        "K": {T: reconstruct(ring, {i: servers[i].key.k_i for i in T}) for T in ((1, 2), (2, 3))},
    }


before = snapshot()
print(table(("server", "epoch", "pk_i = a k_i + 2e_i", "k_i (never sent)", "mu_u"), [
    (f"S_{i}", before["epoch"][i], peek(ring.to_bytes(before["pk"][i])), peek(ring.to_bytes(before["k"][i])), before["mu"][i])
    for i in servers
]))
for T, K in before["K"].items():
    print(f"K from {set(T)}: {coefficients(K)}")

server  epoch  pk_i = a k_i + 2e_i  k_i (never sent)    mu_u
------  -----  -------------------  ------------------  ----
S_1     0      52e1…b706 (57344 B)  317a…cd00 (2048 B)  1
S_2     0      7e04…640b (57344 B)  9bf4…9b01 (2048 B)  2
S_3     0      9827…1000 (57344 B)  056f…6902 (2048 B)  1
K from {1, 2}: [0, -4, 2, 3, -2, 0]..., max |coefficient| = 15
K from {2, 3}: [0, -4, 2, 3, -2, 0]..., max |coefficient| = 15


## Steps 1 to 3: every $S_j$ deals its update shares

$S_j$ samples $\alpha_1, \dots, \alpha_{t-1} \leftarrow R_q$ (here one coefficient polynomial, since $t = 2$) and
computes the commitments $h = \{H(\alpha_k)\}$ with the linear hash $H(x) = A_H x \bmod q$ (plan decision 6).
To each $S_i$ it sends $\{j, \omega, h, E(i, F_{ji})\}$ and $sig_j = Sign(sk_j, (j, \omega, h, E(i, F_{ji})))$,
where $F_{ji} = [F]_j(i)$. The paper has every one of at least $t$ servers deal. Here all $N$ do.

In [5]:
dealt = {j: server.key_update_shares() for j, server in servers.items()}
print(table(("S_j -> S_i", "epoch w", "h_1 = H(alpha_1)", "E(i, F_ji)", "sig_j", "payload"), [
    (f"S_{j} -> S_{i}", share.epoch, peek(share.commitments[0]), peek(share.encrypted_share), peek(share.signature), f"{payload_size(share)} B")
    for j, row in dealt.items()
    for i, share in row.items()
]))
one = dealt[1][2]
print()
print(f"commitments h (t - 1 = {len(one.commitments)} of n * 4 B): {sum(map(len, one.commitments))} B")
print(f"E(i, F_ji) = nonce 12 B + F_ji {4 * pp.n} B + GCM tag 16 B: {len(one.encrypted_share)} B")
print(f"sig_j (ML-DSA-44): {len(one.signature)} B")
print(f"one KeyUpdateShare, bytes fields: {payload_size(one)} B")
print(f"dealt by one server to all {pp.N} servers: {sum(payload_size(s) for s in dealt[1].values())} B")
print(f"all {pp.N * pp.N} shares of the update: {sum(payload_size(s) for row in dealt.values() for s in row.values())} B")

S_j -> S_i  epoch w  h_1 = H(alpha_1)    E(i, F_ji)          sig_j               payload
----------  -------  ------------------  ------------------  ------------------  -------
S_1 -> S_1  0        f5da…3f08 (2048 B)  b1fc…ebe7 (2076 B)  3ade…293a (2420 B)  6544 B
S_1 -> S_2  0        f5da…3f08 (2048 B)  7f5c…744b (2076 B)  3666…3845 (2420 B)  6544 B
S_1 -> S_3  0        f5da…3f08 (2048 B)  2bc9…f204 (2076 B)  bb78…2e3e (2420 B)  6544 B
S_2 -> S_1  0        c852…d80b (2048 B)  bea7…9332 (2076 B)  b751…3246 (2420 B)  6544 B
S_2 -> S_2  0        c852…d80b (2048 B)  806a…69d3 (2076 B)  9849…364b (2420 B)  6544 B
S_2 -> S_3  0        c852…d80b (2048 B)  78f3…b25e (2076 B)  221e…2c37 (2420 B)  6544 B
S_3 -> S_1  0        bd6d…4c02 (2048 B)  4fb3…83a8 (2076 B)  759a…3044 (2420 B)  6544 B
S_3 -> S_2  0        bd6d…4c02 (2048 B)  74b6…8b0c (2076 B)  7bfb…3036 (2420 B)  6544 B
S_3 -> S_3  0        bd6d…4c02 (2048 B)  4b41…73ce (2076 B)  3ce7…2b3b (2420 B)  6544 B

commitments h (t - 1 = 1 of n

## Step 4: $S_i$ checks every received share

For each $S_j$'s share, $S_i$ verifies $sig_j$ under $pk_j^{sig}$, checks $\omega$ is its current epoch,
decrypts $F_{ji}$ and checks the commitment equation
$H(F_{ji}) = \sum_{k=1}^{t-1} i^k H(\alpha_k)$. This holds because $H$ is linear: $H(\sum_k i^k \alpha_k) = \sum_k i^k H(\alpha_k)$.
Nothing is applied yet. The table shows both sides of the equation per (sender, receiver).

In [6]:
checks = {}
for i, server in servers.items():
    for check in server.key_update_check([dealt[j][i] for j in servers]):
        checks[(check.sender, check.receiver)] = check
print(table(("S_j -> S_i", "H(F_ji)", "sum_k i^k h_k", "equal", "outcome"), [
    (f"S_{j} -> S_{i}", peek(c.recomputed), peek(c.committed), c.recomputed == c.committed, c.outcome.name)
    for (j, i), c in checks.items()
]))
print()
print(f"shares accepted: {sum(c.outcome is ShareOutcome.ACCEPTED for c in checks.values())} of {len(checks)}")
print(f"epochs after step 4 (nothing applied): {[s.epoch for s in servers.values()]}")

S_j -> S_i  H(F_ji)             sum_k i^k h_k       equal  outcome
----------  ------------------  ------------------  -----  --------
S_1 -> S_1  f5da…3f08 (2048 B)  f5da…3f08 (2048 B)  True   ACCEPTED
S_2 -> S_1  c852…d80b (2048 B)  c852…d80b (2048 B)  True   ACCEPTED
S_3 -> S_1  bd6d…4c02 (2048 B)  bd6d…4c02 (2048 B)  True   ACCEPTED
S_1 -> S_2  eab5…7f00 (2048 B)  eab5…7f00 (2048 B)  True   ACCEPTED
S_2 -> S_2  90a5…b107 (2048 B)  90a5…b107 (2048 B)  True   ACCEPTED
S_3 -> S_2  b3db…9804 (2048 B)  b3db…9804 (2048 B)  True   ACCEPTED
S_1 -> S_3  df90…be08 (2048 B)  df90…be08 (2048 B)  True   ACCEPTED
S_2 -> S_3  58f8…8a03 (2048 B)  58f8…8a03 (2048 B)  True   ACCEPTED
S_3 -> S_3  a949…e406 (2048 B)  a949…e406 (2048 B)  True   ACCEPTED

shares accepted: 9 of 9
epochs after step 4 (nothing applied): [0, 0, 0]


## Step 5: apply the update

$S_i$ sets $k_i' = k_i + \sum_j F_{ji} \bmod q$ (the printed $\lambda_{i,j}$ is dropped, plan decision 6:
each $[F]_j$ has degree $t - 1$ and constant term $0$, so the sum $\sum_j [F]_j$ is a zero-constant sharing
and $K = \sum_i \lambda_i k_i'$ is unchanged). $S_i$ then draws a fresh $e_i$ for
$pk_i' = a k_i' + 2e_i$, sets $\mu_u = 0$ for every user and starts epoch $\omega + 1$. Users' stored records
$(ID_u, pk_u, \sigma)$ and the outsourced items are not touched.

In [7]:
for i, server in servers.items():
    server.key_update_apply([dealt[j][i] for j in servers])
after = snapshot()
print(table(("server", "epoch", "pk_i before", "pk_i after", "k_i before", "k_i after", "mu_u"), [
    (
        f"S_{i}",
        f"{before['epoch'][i]} -> {after['epoch'][i]}",
        peek(ring.to_bytes(before["pk"][i])),
        peek(ring.to_bytes(after["pk"][i])),
        peek(ring.to_bytes(before["k"][i])),
        peek(ring.to_bytes(after["k"][i])),
        f"{before['mu'][i]} -> {after['mu'][i]}",
    )
    for i in servers
]))
print()
for T in before["K"]:
    print(f"K from {set(T)} before: {coefficients(before['K'][T])}")
    print(f"K from {set(T)} after:  {coefficients(after['K'][T])}  unchanged: {(before['K'][T] == after['K'][T]).all()}")

server  epoch   pk_i before          pk_i after           k_i before          k_i after           mu_u
------  ------  -------------------  -------------------  ------------------  ------------------  ------
S_1     0 -> 1  52e1…b706 (57344 B)  4f26…6b04 (57344 B)  317a…cd00 (2048 B)  14ef…400b (2048 B)  1 -> 0
S_2     0 -> 1  7e04…640b (57344 B)  b58e…ca06 (57344 B)  9bf4…9b01 (2048 B)  61de…8106 (2048 B)  2 -> 0
S_3     0 -> 1  9827…1000 (57344 B)  0bf7…2a09 (57344 B)  056f…6902 (2048 B)  aecd…c201 (2048 B)  1 -> 0

K from {1, 2} before: [0, -4, 2, 3, -2, 0]..., max |coefficient| = 15
K from {1, 2} after:  [0, -4, 2, 3, -2, 0]..., max |coefficient| = 15  unchanged: True
K from {2, 3} before: [0, -4, 2, 3, -2, 0]..., max |coefficient| = 15
K from {2, 3} after:  [0, -4, 2, 3, -2, 0]..., max |coefficient| = 15  unchanged: True


## The user is unaffected (Lemma 3)

alice logs in again with only her password, on both subsets. The servers' shares and the TOPRF noise are
new, but $F = a_x K + 2e''$ is again within the extractor's tolerance of the registration value, so
$K_u = E(F, \sigma)$ and $pk_u$ are the same.

In [8]:
login_12_after = protocol.login(QPASEUser(pp, "alice", PASSWORD), servers, (1, 2))
alice_after = QPASEUser(pp, "alice", PASSWORD)
login_23_after = protocol.login(alice_after, servers, (2, 3))
print(table(("login", "K_u before", "K_u after", "pk_u identical", "K_u identical"), [
    (
        f"{set(ids)}",
        peek(was.k_u),
        peek(now.k_u),
        was.user_pk == now.user_pk,
        was.k_u == now.k_u,
    )
    for ids, was, now in (((1, 2), login_12_before, login_12_after), ((2, 3), login_23_before, login_23_after))
]))
print(f"mu_u after these two logins: {[s.users['alice'].fail_count for s in servers.values()]}")

login   K_u before          K_u after           pk_u identical  K_u identical
------  ------------------  ------------------  --------------  -------------
{1, 2}  f2b1…ae43 (1792 B)  f2b1…ae43 (1792 B)  True            True
{2, 3}  f2b1…ae43 (1792 B)  f2b1…ae43 (1792 B)  True            True
mu_u after these two logins: [1, 2, 1]


A document outsourced before the update is searched and decrypted with keys derived from the same $K_u$.

In [9]:
retrieved = {}
for keyword in ("medical", "travel"):
    run = protocol.retrieve(alice_after, servers, keyword)
    retrieved[keyword] = run.documents
    print(f"Retrieve {keyword!r}: items {[o.item_id for o in run.opened]}, "
          f"returned by {[o.server_ids for o in run.opened]}")
    for o in run.opened:
        print(f"  item {o.item_id}: {o.data.decode()}")

Retrieve 'medical': items [1, 3], returned by [(1,), (3,)]
  item 1: Blood panel, 14 Sep 2026. Haemoglobin 14.1 g/dL, LDL 2.9 mmol/L, HbA1c 5.2%. No follow-up.
  item 3: Lumbar spine MRI: mild disc bulge at L4-L5, no nerve compression. Physiotherapy advised.


Retrieve 'travel': items [2], returned by [(3,)]
  item 2: Booking KX42QP: Mumbai BOM to Lisbon LIS, 3 Nov 2026 09:40, seat 23A, one checked bag.


## Rejected shares leave the server unchanged

A share from an earlier epoch has a valid signature but the wrong $\omega$, so it is rejected as a replay.
A share with a flipped signature bit is rejected by $Ver$. In both cases $S_1$ raises `ProtocolAbort` and
applies nothing, even though the other two shares are honest.

In [10]:
fresh = {j: server.key_update_shares() for j, server in servers.items()}
receiver = servers[1]
state_before = (receiver.epoch, receiver.key.k_i.copy(), receiver.key.pk_i.copy())
replayed = dealt[2][1]
flipped = dataclasses.replace(fresh[2][1], signature=bytes([fresh[2][1].signature[0] ^ 1]) + fresh[2][1].signature[1:])
outcomes = {}
for name, bad in (("replayed epoch-0 share from S_2", replayed), ("flipped signature from S_2", flipped)):
    shares = [fresh[1][1], bad, fresh[3][1]]
    verdicts = {c.sender: c.outcome for c in receiver.key_update_check(shares)}
    try:
        receiver.key_update_apply(shares)
        aborted, reason = False, "applied"
    except ProtocolAbort as error:
        aborted, reason = True, str(error)
    unchanged = (receiver.epoch, bool((receiver.key.k_i == state_before[1]).all()), bool((receiver.key.pk_i == state_before[2]).all()))
    outcomes[name] = (verdicts, aborted, unchanged == (1, True, True))
    print(f"{name}: verdicts {{sender: outcome}} = { {j: o.name for j, o in verdicts.items()} }")
    print(f"  S_1 raised ProtocolAbort: {aborted} ({reason})")
    print(f"  S_1 epoch / k_i unchanged / pk_i unchanged: {unchanged}")

replayed epoch-0 share from S_2: verdicts {sender: outcome} = {1: 'ACCEPTED', 2: 'WRONG_EPOCH', 3: 'ACCEPTED'}
  S_1 raised ProtocolAbort: True (key update share from S_2 rejected: WRONG_EPOCH)
  S_1 epoch / k_i unchanged / pk_i unchanged: (1, True, True)


flipped signature from S_2: verdicts {sender: outcome} = {1: 'ACCEPTED', 2: 'BAD_SIGNATURE', 3: 'ACCEPTED'}
  S_1 raised ProtocolAbort: True (key update share from S_2 rejected: BAD_SIGNATURE)
  S_1 epoch / k_i unchanged / pk_i unchanged: (1, True, True)


## A second update through the protocol runner

`protocol.key_update` runs steps 1 to 5 for all servers. It checks every share at every receiver first,
so one bad share aborts the whole update before any server moves to the next epoch.

In [11]:
second = protocol.key_update(servers)
login_after_second = protocol.login(QPASEUser(pp, "alice", PASSWORD), servers, (1, 2))
print(f"epochs: {[s.epoch for s in servers.values()]}")
print(f"shares accepted: {sum(c.outcome is ShareOutcome.ACCEPTED for c in second.checks.values())} of {len(second.checks)}")
print(f"K_u after the second update: {peek(login_after_second.k_u)}, identical to Register: {login_after_second.k_u == registered.k_u}")

epochs: [2, 2, 2]
shares accepted: 9 of 9
K_u after the second update: f2b1…ae43 (1792 B), identical to Register: True


## Assertions

The run fails here unless every property held.

In [12]:
assert all(before["epoch"][i] == 0 and after["epoch"][i] == 1 for i in servers), "epoch 0 -> 1"
assert all((before["k"][i] != after["k"][i]).any() for i in servers), "every k_i changed"
assert all((before["pk"][i] != after["pk"][i]).any() for i in servers), "every pk_i changed"
assert all(before["mu"][i] > 0 for i in servers) and all(after["mu"][i] == 0 for i in servers), "mu_u reset to 0 everywhere"
assert all((before["K"][T] == after["K"][T]).all() for T in before["K"]), "master key K unchanged (Lemma 3)"
assert len(checks) == pp.N * pp.N and all(c.outcome is ShareOutcome.ACCEPTED for c in checks.values())
assert all(c.recomputed == c.committed for c in checks.values()), "H(F_ji) == sum_k i^k h_k for every share"
for was, now in ((login_12_before, login_12_after), (login_23_before, login_23_after)):
    assert now.user_pk == was.user_pk == registered.key_message.user_pk, "same pk_u"
    assert now.k_u == was.k_u == registered.k_u, "same K_u"
expected = {kw: [d for _, k, _, d in DOCUMENTS if k == kw] for kw in ("medical", "travel")}
assert retrieved == expected, "pre-update documents retrieved byte-equal"
for name, (verdicts, aborted, unchanged) in outcomes.items():
    assert aborted and unchanged, f"{name}: rejected all-or-nothing"
    assert verdicts[1] is ShareOutcome.ACCEPTED and verdicts[3] is ShareOutcome.ACCEPTED, name
assert outcomes["replayed epoch-0 share from S_2"][0][2] is ShareOutcome.WRONG_EPOCH
assert outcomes["flipped signature from S_2"][0][2] is ShareOutcome.BAD_SIGNATURE
assert all(s.epoch == 2 for s in servers.values()) and login_after_second.k_u == registered.k_u
print("all key update properties held")

all key update properties held
